In [ ]:
import mysql.connector

try:
    connection = mysql.connector.connect(
        host="localhost",      
        user="root",        
        password="35459583",
        database="hospital_db"
    )
    if connection.is_connected():
        print("Successfully connected to the database!")

except Exception as e:
    print(f"Error: {e}")

In [ ]:
input_raw = input("Please enter your target city or state (e.g., CA): ").strip()

query = f"""
WITH avg_drg_billing AS (
    SELECT drg_code, AVG(avg_total_payment) AS avg_payment
    FROM billing
    GROUP BY drg_code
)
SELECT 
    h.hospital_name, r.drg_code, r.drg_description, h.rating AS star_rating,
    b.avg_total_payment AS hospital_price,
    round(avg_drg_billing.avg_payment, 2) AS national_avg_price,
    round(avg_drg_billing.avg_payment - b.avg_total_payment, 2) AS total_savings,
    l.city, l.state, l.state_full
FROM hospitals h
LEFT JOIN billing b ON h.facility_id = b.facility_id
LEFT JOIN locations l ON h.location_id = l.location_id
LEFT JOIN ref_drg r ON b.drg_code = r.drg_code
LEFT JOIN avg_drg_billing ON b.drg_code = avg_drg_billing.drg_code
WHERE 
    h.rating = 5 
    AND b.avg_total_payment < avg_drg_billing.avg_payment 
    AND (l.state = '{input_raw}' OR l.city = '{input_raw}' OR l.state_full = '{input_raw}')
ORDER BY total_savings DESC
LIMIT 20;
"""

In [ ]:
cursor = connection.cursor()
cursor.execute(query)
results = cursor.fetchall()

In [ ]:
if connection.is_connected():
    cursor.close()
    connection.close()

In [ ]:
import pandas as pd
columns = [
    'Hospital Name', 
    'DRG Code', 
    'DRG Description', 
    'Star Rating', 
    'Hospital Price', 
    'National Avg Price', 
    'Total Savings', 
    'City', 
    'State', 
    'Full State Name'
]

df = pd.DataFrame(results, columns=columns)
display(df)